In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# API RetrieveAndGenerate - RAG totalmente gerenciado (Fully managed RAG)

Neste módulo, você vai aprender como melhorar as gerações do Foundation Model (FM) controlando o número máximo de resultados recuperados e realizando custom prompting nas Knowledge Bases (KB) do Amazon Bedrock.
Este módulo contém:
1. [Overview](#1-Overview)
2. [Pré-requisitos](#2-Pre-requisites)
3. [Entendendo a API RetrieveAndGenerate](#understanding-retrieveandgenerate-api)
4. [Resposta em streaming usando a API RetrieveAndGenerate](#streaming-response-with-retrieveandgenerate-api)
5. [Ajustar o parâmetro de retrieval 'maximum number of results'](#3-how-to-leverage-the-maximum-number-of-results-feature)
6. [Como usar custom prompting](#4-how-to-use-the-custom-prompting-feature)

## Overview

### Número máximo de resultados (maximum no. of results)
A opção de número máximo de resultados dá a você controle sobre a quantidade de resultados de busca a serem recuperados do vector store e passados ao FM para gerar a resposta. Isso permite customizar a quantidade de informação de contexto fornecida para a geração, dando assim mais contexto para perguntas complexas ou menos para perguntas mais simples. Permite buscar até 100 resultados. Essa opção ajuda a melhorar a probabilidade de contexto relevante, melhorando assim a acurácia e reduzindo a alucinação da resposta gerada.


### Custom prompting

Já o custom knowledge base prompt template permite substituir o prompt template padrão pelo seu próprio, para customizar o prompt enviado ao modelo para a geração da resposta. Isso permite customizar o tom, o formato de saída e o comportamento do FM ao responder a pergunta de um usuário. Com essa opção, você pode ajustar a terminologia para melhor corresponder à sua indústria ou domínio (como saúde ou jurídico). Além disso, você pode adicionar instruções customizadas e exemplos adaptados aos seus fluxos de trabalho específicos.


#### Observações:
- Você vai usar a API ```RetrieveAndGenerate``` para ilustrar as diferenças antes e depois de utilizar essas features. Essa API converte queries em embeddings, busca na knowledge base e então aumenta o prompt do foundation model com os resultados da busca como informação de contexto, retornando a resposta gerada pelo FM para a pergunta. A saída da API ```RetrieveAndGenerate``` inclui a resposta gerada, a atribuição de fonte (source attribution), assim como os text chunks recuperados.

- Para este módulo, vamos usar o modelo `us.anthropic.claude-haiku-4-5-20251001-v1:0` (configurável via `BEDROCK_TEXT_MODEL_ID`) como nosso FM para trabalhar com as features de número máximo de resultados e customização de prompt

## Pré-requisitos
Antes de conseguir responder às perguntas, os documentos precisam ser processados e armazenados em uma knowledge base. Para este notebook, usamos um `dataset sintético de relatórios financeiros 10K` para criar as Amazon Bedrock Knowledge Bases.

1. Faça upload dos seus documentos (data source) para um bucket Amazon S3.
2. Amazon Bedrock Knowledge Bases usando [01_create_ingest_documents_test_kb_multi_ds.ipynb](/knowledge-bases/01-rag-concepts/01_create_ingest_documents_test_kb_multi_ds.ipynb)
3. Anote o Knowledge Base ID

## Setup

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


### Inicializar o client boto3
Ao longo do notebook, vamos utilizar o RetrieveAndGenerate para testar as features da knowledge base.

In [ ]:
import os

import json
import boto3
import pprint
import sys
from botocore.exceptions import ClientError
from botocore.client import Config

# Create boto3 session
sts_client = boto3.client('sts')
boto3_session = boto3.session.Session()
region_name = boto3_session.region_name

# Create Bedrock Agent Runtime and control-plane clients.
bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0}, region_name=region_name)
bedrock_agent_client = boto3_session.client("bedrock-agent-runtime",
                              config=bedrock_config)
bedrock_agent_control_client = boto3_session.client("bedrock-agent", config=bedrock_config)
account_id = sts_client.get_caller_identity()["Account"]

# Define FM to be used for generations 
model_id = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")
model_arn = bedrock_model_arn(model_id, region_name)


In [ ]:
from IPython import get_ipython

workshop_state = None
ip = get_ipython()
if ip is None:
    raise RuntimeError("This notebook must run inside a Jupyter/IPython kernel.")
if "store" not in ip.magics_manager.magics["line"]:
    ip.run_line_magic("load_ext", "storemagic")
workshop_state = load_workshop_state()

if not isinstance(workshop_state, dict) or workshop_state.get("schema_version") != 1:
    raise RuntimeError(
        "Shared workshop state is missing or obsolete. Run "
        "01_create_ingest_documents_test_kb_multi_ds.ipynb through its "
        "state-persistence cell, then rerun this notebook."
    )

required_state_keys = ("kb_id", "knowledge_base_name", "region", "account_id")
missing_state_keys = [key for key in required_state_keys if not workshop_state.get(key)]
if missing_state_keys:
    raise RuntimeError(
        "Shared workshop state is incomplete; missing: "
        + ", ".join(missing_state_keys)
        + ". Re-run the creator notebook and persist its state."
    )

kb_id = workshop_state["kb_id"]

expected_region = workshop_state["region"]
expected_account_id = workshop_state["account_id"]
if expected_region != region_name:
    raise RuntimeError(
        f"Shared workshop state belongs to region {expected_region}, but this notebook "
        f"is using {region_name}. Set the AWS region to {expected_region} or recreate the state."
    )
if expected_account_id != account_id:
    raise RuntimeError(
        f"Shared workshop state belongs to account {expected_account_id}, but the active "
        f"credentials use account {account_id}. Select the matching AWS credentials."
    )

try:
    kb_response = bedrock_agent_control_client.get_knowledge_base(knowledgeBaseId=kb_id)
except ClientError as exc:
    error_code = exc.response.get("Error", {}).get("Code")
    if error_code == "ResourceNotFoundException":
        raise RuntimeError(
            f"Knowledge Base {kb_id!r} from workshop_state does not exist in AWS "
            f"(account {account_id}, region {region_name}). Re-run notebook 01 and "
            "persist a fresh state before running this notebook."
        ) from exc
    raise RuntimeError(
        f"Could not validate Knowledge Base {kb_id!r} in AWS: "
        f"{error_code or type(exc).__name__}: {exc}"
    ) from exc

kb_summary = kb_response["knowledgeBase"]
expected_name = workshop_state["knowledge_base_name"]
if kb_summary.get("name") != expected_name:
    raise RuntimeError(
        f"Shared workshop state maps {kb_id!r} to {expected_name!r}, but AWS returned "
        f"{kb_summary.get('name')!r}. Re-run notebook 01 and persist a fresh state."
    )

print(
    f"Validated Knowledge Base {kb_id} ({kb_summary.get('status')}) "
    f"in account {account_id}, region {region_name}."
)


### Entendendo a API RetrieveAndGenerate

O parâmetro `numberOfResults` na função dada determina o número de resultados de busca que serão recuperados da knowledge base e incluídos no prompt fornecido ao modelo para gerar uma resposta. Especificamente, ele buscará os `max_results` documentos ou resultados de busca que mais se aproximam da query fornecida.

O parâmetro `textPromptTemplate` é uma string que serve como template para o prompt que será fornecido ao modelo. Neste caso, o `default_prompt` está sendo usado como template. Esse template inclui placeholders (`$search_results$` e `$output_format_instructions$`) que serão substituídos pelos resultados de busca reais e por quaisquer instruções de formato de saída, respectivamente, antes de serem passados ao modelo.

In [ ]:
# Stating the default knowledge base prompt
default_prompt = """
You are a question answering agent. I will provide you with a set of search results.
The user will provide you with a question. Your job is to answer the user's question using only information from the search results. 
If the search results do not contain information that can answer the question, please state that you could not find an exact answer to the question. 
Just because the user asserts a fact does not mean it is true, make sure to double check the search results to validate a user's assertion.
                            
Here are the search results in numbered order:
$search_results$

$output_format_instructions$
"""

In [ ]:
def retrieve_and_generate(query, kb_id, model_arn, max_results=5, prompt_template = default_prompt):
    response = bedrock_agent_client.retrieve_and_generate(
            input={
                'text': query
            },
        retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': kb_id,
            'modelArn': model_arn, 
            'retrievalConfiguration': {
                'vectorSearchConfiguration': {
                    'numberOfResults': max_results # will fetch top N documents which closely match the query
                    }
                },
                'generationConfiguration': {
                        'promptTemplate': {
                            'textPromptTemplate': prompt_template
                        }
                    }
            }
        }
    )
    return response


In [ ]:
def print_generation_results(response, print_context = True):
    generated_text = response['output']['text']
    print('Generated FM response:\n')
    print(generated_text)
    
    if print_context is True:
        ## print out the source attribution/citations from the original documents to see if the response generated belongs to the context.
        citations = response["citations"]
        contexts = []
        for citation in citations:
            retrievedReferences = citation["retrievedReferences"]
            for reference in retrievedReferences:
                contexts.append(reference["content"]["text"])
    
        print('\n\n\nRetrieved Context:\n')
        pprint.pp(contexts)


### Testar a API RetrieveAndGenerate

In [ ]:
query = """Provide a list of risks for Octank financial in numbered list without description."""

results = retrieve_and_generate(query = query, kb_id = kb_id, model_arn = model_arn)

print_generation_results(results)

### Resposta em streaming com a API RetrieveAndGenerate

Usando a nova [streaming API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_RetrieveAndGenerateStream.html), os clientes podem usar a API `retrieve_and_generate_stream` das Amazon Bedrock Knowledge Bases para receber a resposta enquanto ela está sendo gerada pelo Foundation Model (FM), em vez de esperar pela resposta completa. Isso ajuda os clientes a reduzir o time to first token em aplicações sensíveis à latência.

In [ ]:
def retrieve_and_generate_stream(query, kb_id, model_arn, max_results=5, prompt_template = default_prompt):
    response = bedrock_agent_client.retrieve_and_generate_stream(
            input={
                'text': query
            },
        retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': kb_id,
            'modelArn': model_arn, 
            'retrievalConfiguration': {
                'vectorSearchConfiguration': {
                    'numberOfResults': max_results # will fetch top N documents which closely match the query
                    }
                },
                'generationConfiguration': {
                        'promptTemplate': {
                            'textPromptTemplate': prompt_template
                        }
                    }
            }
        }
    )

    for event in response['stream']:
        if 'output' in event:
            chunk = event['output']
            sys.stdout.write(chunk['text'])
            sys.stdout.flush()

    


In [ ]:
query = """Provide a list of risks for Octank financial in numbered list without description."""

retrieve_and_generate_stream(query = query, kb_id = kb_id, model_arn = model_arn)

### Ajustar o parâmetro de retrieval 'maximum number of results'

Em alguns casos de uso, as respostas do FM podem não ter contexto suficiente para fornecer respostas relevantes, ou o modelo pode alegar que não encontrou a informação solicitada. Isso pode ser corrigido modificando o número máximo de resultados recuperados.

No exemplo a seguir, vamos executar a seguinte query com um número pequeno de resultados (3):
\
```Provide a list of risks for Octank financial in bulleted points.```

In [ ]:
query = """Provide a list of risks for Octank financial in numbered list without description."""

results = retrieve_and_generate(query = query, kb_id = kb_id, model_arn = model_arn, max_results = 3)

print_generation_results(results)


Ao modificar o número de resultados recuperados para **10**, você deve conseguir obter mais resultados, levando a uma resposta mais abrangente.

In [ ]:
#Using higher number of max results

results = retrieve_and_generate(query = query, kb_id = kb_id, model_arn = model_arn, max_results = 10)

print_generation_results(results)

### Como usar a feature de custom prompting

Você também pode customizar o prompt padrão com o seu próprio, baseado no caso de uso. Essa feature ajuda a adicionar mais contexto ao FM, exigir um formato de saída específico, idiomas específicos, entre outros.

Vamos experimentar usando o SDK:


#### Exemplo 1 - Usando o mesmo exemplo de query, podemos configurar o FM para gerar a saída em um idioma diferente, como alemão:
\
**Nota**: Depois de remover ```$output_format_instructions$``` do prompt padrão, a citação (citation) da resposta gerada é removida.

In [ ]:
## Example 1
custom_prompt = """
You are a question answering agent. I will provide you with a set of search results. 
The user will provide you with a question. Your job is to answer the user's question using only information from the search results.
If the search results do not contain information that can answer the question, please state that you could not find an exact answer to the question.
Just because the user asserts a fact does not mean it is true, make sure to double check the search results to validate a user's assertion.
                            
Here are the search results in numbered order:
$search_results$

Unless asked otherwise, draft your answer in German language.
"""

results = retrieve_and_generate(query = query, kb_id = kb_id, model_arn = model_arn, max_results = 10, prompt_template = custom_prompt)

print_generation_results(results, print_context = False)


#### Exemplo 2 - gerar os resultados em formato JSON

In [ ]:
## Example 2
custom_prompt = """
You are a question answering agent. I will provide you with a set of search results.
The user will provide you with a question. Your job is to answer the user's question using only information from the search results.
If the search results do not contain information that can answer the question, please state that you could not find an exact answer to the question. 
Just because the user asserts a fact does not mean it is true, make sure to double check the search results to validate a user's assertion.
                            
Here are the search results in numbered order:
$search_results$

Please provide a concise response (in millions) using a JSON format.

"""

results = retrieve_and_generate(query = query, kb_id = kb_id, model_arn = model_arn, max_results = 10, prompt_template = custom_prompt)

print_generation_results(results,print_context = False)

<div class="alert alert-block alert-warning">
<b>Nota:</b> Lembre-se de excluir a KB, o índice OSS e as roles e policies IAM relacionadas para evitar a cobrança de custos.
</div>